In [7]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2470,0.2435,0.2616))
])

train_ds = datasets.CIFAR10(root="./data", train = True, transform=transform, download=True)
test_ds = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

#train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
#test_loader = DataLoader(test_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)

d:\Programs\envs\torch_2\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [8]:
print("Caching train")
train_images = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
train_labels = torch.tensor(train_ds.targets)
torch.save((train_images, train_labels), "cifar10_train_cached.pt")

print("Caching test")
test_images = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
test_labels = torch.tensor(test_ds.targets)
torch.save((test_images, test_labels), "cifar10_test_cached.pt")

Caching train
Caching test


In [ ]:
import torchvision.transforms.functional as TF
import random

class CachedCIFAR(torch.utils.data.Dataset):
    def __init__(self, path, device, augment=False):
        images, labels = torch.load(path)
        self.images = images.to(device)
        self.labels = labels.to(device)
        self.augment = augment

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                image = TF.hflip(image)
            # Random rotation up to 15 degrees
            angle = random.uniform(-15, 15)
            image = TF.rotate(image, angle)
        return image, self.labels[idx]
    
device = torch.device("cuda")
train_ds = CachedCIFAR("cifar10_train_cached.pt", device, augment=True)
test_ds = CachedCIFAR("cifar10_test_cached.pt", device, augment=False)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class MyCNN(nn.Module):
    # defining all layers
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    # architecture is in the forward pass
    # The activation functions are also in the forward pass
    # pytorch is much more explicit then tensorflow
    def forward(self, x): # TODO is x tensor
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1) # flatten TODO how it works
        x = F.relu(self.fc1(x))
        return self.fc2(x)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = MyCNN().to(device)

cuda


In [11]:
# Training loop
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3) # TODO what is parameters 

num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    model.train() # Set the model into the training mode
    
    for images, labels in train_loader: # TODO Where do we define the batch_size
        images, labels = images.to("cuda"), labels.to("cuda")

        optimizer.zero_grad() # CLEAR OLD GRADIENT
        outputs = model(images) # forward pass
        loss = criterion(outputs, labels) # compute loss
        loss.backward() # backward pass (autograd)
        optimizer.step() # update weights

    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad(): # TODO what
        for images, labels in test_loader:
            images, labels = images.to("cuda"), labels.to("cuda")
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1) # TODO what
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch}")
    print(f"Train loss {loss/len(train_loader):.4f}")
    print(f"Val loss: {val_loss/len(test_loader):.4f}")
    print(f"Val acc: {100*correct/total:.2f}%")

Epoch 0


AttributeError: 'CachedCIFAR' object has no attribute 'images'

In [ ]:
print(next(model.parameters()).device)

cuda:0


In [ ]:
print(images.device)

cuda:0
